# WM-811K 실제 test 대표 샘플 추출

대시보드 시연용 실제 웨이퍼를 클래스별 4~5개씩 Google Drive의 전처리 배열에서 꺼냅니다. 현재 manifest는 9개 클래스 × 5개 = 45개입니다. GPU는 필요하지 않으며 CPU 런타임에서 보통 1분 이내에 끝납니다.

샘플은 모델 선택이 끝난 뒤 test 예측에서 정한 정성적 예시입니다. 모델 선택이나 성능 계산에는 사용하지 않습니다. 클래스마다 고신뢰 정답, 전형적 정답(신뢰도 중앙값 근처), 경계 근처 정답(검토 임계값 근처), 검토 대상 정답, 대표 오분류를 고르며, 해당 사례가 없는 클래스는 4개가 됩니다.

로컬에 `wafer_maps_64.npy`가 있으면 같은 추출을 `코드/wm811k/export_wm811k_demo_samples.py`로 실행할 수 있습니다.

## 1. Drive 연결과 manifest 업로드

파일 선택 창에서 프로젝트의 `결과물/wm811k/selected_model_results/demo_sample_manifest.json` 하나를 선택하세요.

In [ ]:
from google.colab import drive, files
from pathlib import Path
from collections import Counter
import hashlib, json, shutil
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np

drive.mount('/content/drive')
uploaded = files.upload()
assert 'demo_sample_manifest.json' in uploaded, 'demo_sample_manifest.json을 선택하세요.'
manifest = json.loads(uploaded['demo_sample_manifest.json'].decode('utf-8'))
assert manifest['source_split'] == 'test'
assert manifest['used_for_model_selection'] is False
assert manifest['sample_count'] == len(manifest['samples'])
assert len({item['array_index'] for item in manifest['samples']}) == manifest['sample_count']
class_counts = Counter(item['true_class'] for item in manifest['samples'])
assert len(class_counts) == 9, f'9개 클래스가 모두 필요합니다: {dict(class_counts)}'
assert all(4 <= count <= 5 for count in class_counts.values()), f'클래스별 4~5개가 아닙니다: {dict(class_counts)}'

DRIVE_DATA_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_processed')
MAPS_PATH = DRIVE_DATA_DIR / 'wafer_maps_64.npy'
LABELS_PATH = DRIVE_DATA_DIR / 'labels.npy'
assert MAPS_PATH.is_file(), f'전처리 웨이퍼 배열이 없습니다: {MAPS_PATH}'
assert LABELS_PATH.is_file(), f'라벨 배열이 없습니다: {LABELS_PATH}'
print('manifest:', manifest['sample_count'], '개', dict(sorted(class_counts.items())))
print('웨이퍼 배열:', MAPS_PATH)

## 2. 실제 웨이퍼 추출 및 검증

In [ ]:
maps = np.load(MAPS_PATH, mmap_mode='r', allow_pickle=False)
labels = np.load(LABELS_PATH, mmap_mode='r', allow_pickle=False)
assert len(maps) == len(labels)

OUTPUT_DIR = Path('/content/wm811k_demo_samples')
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
SAMPLES_DIR = OUTPUT_DIR / 'samples'
SAMPLES_DIR.mkdir(parents=True)

exported = []
for item in manifest['samples']:
    index = int(item['array_index'])
    assert 0 <= index < len(maps), f'배열 인덱스 범위 오류: {index}'
    wafer = np.asarray(maps[index], dtype=np.uint8)
    assert wafer.shape == (64, 64), f'웨이퍼 크기 오류: {wafer.shape}'
    assert set(np.unique(wafer)).issubset({0, 1, 2}), f'웨이퍼 값 오류: {np.unique(wafer)}'
    assert int(labels[index]) == int(item['true_label_id']), f'라벨 불일치: {index}'
    npy_name = f"{item['sample_id']}.npy"
    csv_name = f"{item['sample_id']}.csv"
    np.save(SAMPLES_DIR / npy_name, wafer, allow_pickle=False)
    np.savetxt(SAMPLES_DIR / csv_name, wafer, fmt='%d', delimiter=',')
    exported_item = dict(item)
    exported_item['npy_file'] = f'samples/{npy_name}'
    exported_item['csv_file'] = f'samples/{csv_name}'
    exported_item['wafer_sha256'] = hashlib.sha256(wafer.tobytes()).hexdigest()
    exported.append(exported_item)

export_manifest = dict(manifest)
export_manifest['samples'] = exported
export_manifest['exported_map_shape'] = [64, 64]
export_manifest['allowed_values'] = [0, 1, 2]
(OUTPUT_DIR / 'manifest.json').write_text(
    json.dumps(export_manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)
(OUTPUT_DIR / 'README.txt').write_text(
    f'WM-811K 실제 test 대표 샘플 {len(exported)}개 (클래스별 4~5개)\n'
    '각 샘플은 NPY와 CSV 두 형식으로 저장됩니다.\n'
    '성능 평가나 모델 선택이 아닌 대시보드 정성 시연 전용입니다.\n',
    encoding='utf-8',
)
print('추출 완료:', len(exported), '개')

## 3. 샘플 미리보기

In [ ]:
case_titles = {
    'high_confidence_correct': 'high-confidence correct',
    'typical_correct': 'typical correct',
    'boundary_correct': 'near review threshold',
    'low_confidence_correct': 'review correct',
    'representative_error': 'representative error',
}
case_order = list(case_titles)
class_order = list(dict.fromkeys(item['true_class'] for item in exported))
cmap = ListedColormap(['#141821', '#67A479', '#DC4848'])
fig, axes = plt.subplots(len(class_order), len(case_order), figsize=(15, 3 * len(class_order)))
for axis in axes.ravel():
    axis.axis('off')
for item in exported:
    axis = axes[class_order.index(item['true_class']), case_order.index(item['selection_case'])]
    wafer = np.load(OUTPUT_DIR / item['npy_file'], allow_pickle=False)
    axis.imshow(wafer, cmap=cmap, vmin=0, vmax=2, interpolation='nearest')
    axis.set_title(
        f"{item['true_class']} | {case_titles[item['selection_case']]}\n"
        f"pred={item['predicted_class']}, conf={item['calibrated_confidence']:.1%}",
        fontsize=9,
    )
plt.tight_layout()
plt.show()

## 4. ZIP 저장 및 다운로드

다운로드된 `wm811k_demo_samples.zip`을 로컬 프로젝트의 `결과물/wm811k/colab_가져오기/` 폴더에 압축 상태로 넣으세요.

In [ ]:
ZIP_BASE = Path('/content/wm811k_demo_samples')
ZIP_PATH = Path(shutil.make_archive(str(ZIP_BASE), 'zip', root_dir=OUTPUT_DIR))
zip_sha256 = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
DRIVE_BACKUP_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_demo')
DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(ZIP_PATH, DRIVE_BACKUP_DIR / ZIP_PATH.name)
print('ZIP:', ZIP_PATH, f'{ZIP_PATH.stat().st_size / 1024:.1f} KB')
print('SHA-256:', zip_sha256)
print('Drive 백업:', DRIVE_BACKUP_DIR / ZIP_PATH.name)
files.download(str(ZIP_PATH))